# 04. User×Item Matrix & Sparse Representation

추천 시스템의 핵심 입력은 **유저-아이템 상호작용 행렬**입니다.

우리 데이터에서:
- **User** = `actor_id` (GitHub 유저)
- **Item** = `repo_id` (GitHub 레포)
- **Value** = 가중 점수 (이벤트별 가중치 적용)

전체 데이터는 **수백만 유저 × 수백만 레포**로 Dense matrix는 메모리에 올릴 수 없습니다.  
샘플링 후 Dense와 Sparse의 차이를 비교합니다.

## 1. Feedback 데이터 집계

In [1]:
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

from ghrec.recommend import load_period

OUTPUT_DIR = Path("../../data/daily_agg")

WEIGHTS = {
    "WatchEvent": 1.0,
    "ForkEvent": 2.0,
    "IssuesEvent": 0.5,
    "PullRequestEvent": 3.0,
    "IssueCommentEvent": 0.3,
    "PushEvent": 0.2,
}

# 전체 기간 로드
df = load_period(OUTPUT_DIR, date(2026, 2, 15), date(2026, 3, 14))

# 이벤트별 가중치 적용 → 유저-레포 점수 집계
df["score"] = df["type"].map(WEIGHTS).fillna(0) * df["cnt"]
feedback = df.groupby(["actor_id", "repo_id"])["score"].sum().reset_index()
feedback = feedback[feedback["score"] > 0]

n_users = feedback["actor_id"].nunique()
n_items = feedback["repo_id"].nunique()

print(f"전체 feedback 레코드: {len(feedback):,}")
print(f"유니크 유저: {n_users:,}")
print(f"유니크 레포: {n_items:,}")
print(f"\n→ Full matrix 크기: {n_users:,} × {n_items:,} = {n_users * n_items:,} cells")
print(f"→ 실제 non-zero: {len(feedback):,}")
print(f"→ Sparsity: {1 - len(feedback) / (n_users * n_items):.6%}")

전체 feedback 레코드: 12,101,260
유니크 유저: 5,349,865
유니크 레포: 8,474,798

→ Full matrix 크기: 5,349,865 × 8,474,798 = 45,339,025,202,270 cells
→ 실제 non-zero: 12,101,260
→ Sparsity: 99.999973%


## 2. Dense Matrix (샘플링)

활발한 유저 500명 × 인기 레포 200개를 샘플링해서 Dense matrix를 만듭니다.

In [2]:
N_USERS, N_ITEMS = 500, 200

# 인터랙션이 많은 유저/레포를 샘플링 (빈 행/열 줄이기)
top_users = feedback.groupby("actor_id")["score"].count().nlargest(N_USERS).index
top_items = feedback.groupby("repo_id")["score"].count().nlargest(N_ITEMS).index

sample = feedback[
    feedback["actor_id"].isin(top_users) & feedback["repo_id"].isin(top_items)
].copy()

# Dense matrix (pivot)
dense_matrix = sample.pivot_table(
    index="actor_id", columns="repo_id", values="score", fill_value=0
)

n_nonzero = (dense_matrix > 0).sum().sum()
total_cells = dense_matrix.shape[0] * dense_matrix.shape[1]

print(f"Dense matrix shape: {dense_matrix.shape}")
print(f"Non-zero entries: {n_nonzero:,} / {total_cells:,}")
print(f"Sparsity: {1 - n_nonzero / total_cells:.2%}")
print()
dense_matrix.iloc[:10, :8]

Dense matrix shape: (166, 194)
Non-zero entries: 1,354 / 32,204
Sparsity: 95.80%



repo_id,2325298,4542716,12888993,15634981,21289110,21737465,36633370,41881900
actor_id,,,,,,,,
217,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3664,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
93783,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
178424,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
185732,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
234114,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
592534,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
651111,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
713559,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 3. Sparse Matrix (CSR)

같은 데이터를 `scipy.sparse.csr_matrix`로 변환합니다.  
Sparse format은 **non-zero 값만 저장**하므로, sparsity가 높을수록 메모리 절약이 큽니다.

CSR (Compressed Sparse Row) 구조:
- `data`: non-zero 값들
- `indices`: 각 값의 열 인덱스
- `indptr`: 각 행이 시작되는 위치

In [3]:
# Dense → Sparse 변환
dense_values = dense_matrix.values.astype(np.float64)
sparse_matrix = sparse.csr_matrix(dense_values)

print(f"Sparse matrix shape: {sparse_matrix.shape}")
print(f"Non-zero entries (nnz): {sparse_matrix.nnz:,}")
print()

# 내부 구조 살펴보기
print("=== CSR 내부 구조 ===")
print(f"data   (non-zero 값):  {sparse_matrix.data[:10]}  ... ({len(sparse_matrix.data):,}개)")
print(f"indices (열 인덱스):   {sparse_matrix.indices[:10]}  ... ({len(sparse_matrix.indices):,}개)")
print(f"indptr  (행 포인터):   {sparse_matrix.indptr[:10]}  ... ({len(sparse_matrix.indptr):,}개)")
print()

# Sparse matrix 일부 출력
print("=== Sparse matrix 일부 (처음 5행) ===")
print(sparse_matrix[:5])

Sparse matrix shape: (166, 194)
Non-zero entries (nnz): 1,354

=== CSR 내부 구조 ===
data   (non-zero 값):  [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]  ... (1,354개)
indices (열 인덱스):   [ 51  62 111 114 132 161 169 173 188  87]  ... (1,354개)
indptr  (행 포인터):   [ 0  9 11 12 18 35 39 54 75 78]  ... (167개)

=== Sparse matrix 일부 (처음 5행) ===
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 35 stored elements and shape (5, 194)>
  Coords	Values
  (0, 51)	1.0
  (0, 62)	1.0
  (0, 111)	1.0
  (0, 114)	1.0
  (0, 132)	1.0
  (0, 161)	1.0
  (0, 169)	1.0
  (0, 173)	1.0
  (0, 188)	1.0
  (1, 87)	1.0
  (1, 132)	1.0
  (2, 157)	2.0
  (3, 114)	1.0
  (3, 154)	1.0
  (3, 156)	1.0
  (3, 180)	1.0
  (3, 185)	1.0
  (3, 189)	1.0
  (4, 0)	1.0
  (4, 11)	1.0
  (4, 23)	1.0
  (4, 28)	1.0
  (4, 33)	1.0
  (4, 70)	1.0
  (4, 91)	1.0
  (4, 98)	1.0
  (4, 106)	1.0
  (4, 112)	1.0
  (4, 119)	1.0
  (4, 151)	1.0
  (4, 157)	1.0
  (4, 163)	1.0
  (4, 183)	1.0
  (4, 186)	1.0
  (4, 193)	1.0


## 4. 메모리 비교: Dense vs Sparse

In [4]:
def fmt_bytes(b):
    if b < 1024:
        return f"{b} B"
    elif b < 1024**2:
        return f"{b / 1024:.1f} KB"
    elif b < 1024**3:
        return f"{b / 1024**2:.1f} MB"
    else:
        return f"{b / 1024**3:.1f} GB"

# 샘플 데이터
dense_bytes = dense_values.nbytes
sparse_bytes = (sparse_matrix.data.nbytes
                + sparse_matrix.indices.nbytes
                + sparse_matrix.indptr.nbytes)

print(f"{'':>20} {'Dense':>12} {'Sparse (CSR)':>14}")
print(f"{'─' * 48}")
print(f"{'Shape':>20} {str(dense_matrix.shape):>12} {str(sparse_matrix.shape):>14}")
print(f"{'메모리':>20} {fmt_bytes(dense_bytes):>12} {fmt_bytes(sparse_bytes):>14}")
print(f"{'압축률':>20} {'—':>12} {sparse_bytes / dense_bytes:.1%}".ljust(48))
print()
print(f"→ Sparse는 Dense 대비 {dense_bytes / sparse_bytes:.1f}배 적은 메모리 사용")

# 전체 데이터에 대한 추정
full_dense_bytes = n_users * n_items * 8  # float64
full_sparse_bytes = len(feedback) * (8 + 4) + (n_users + 1) * 4  # data + indices + indptr

print(f"\n{'=' * 48}")
print(f"전체 데이터 추정 ({n_users:,} × {n_items:,})")
print(f"{'=' * 48}")
print(f"Dense:  {fmt_bytes(full_dense_bytes)}  ← 메모리에 안 올라감!")
print(f"Sparse: {fmt_bytes(full_sparse_bytes)}")
print(f"압축률: {full_sparse_bytes / full_dense_bytes:.8%}")

                            Dense   Sparse (CSR)
────────────────────────────────────────────────
               Shape   (166, 194)     (166, 194)
                 메모리     251.6 KB        16.5 KB
                 압축률            — 6.6%          

→ Sparse는 Dense 대비 15.2배 적은 메모리 사용

전체 데이터 추정 (5,349,865 × 8,474,798)
Dense:  337802.1 GB  ← 메모리에 안 올라감!
Sparse: 158.9 MB
압축률: 0.00004594%
